# 02 Checkpoint verification

Choose one saved model run, load its checkpoint, validate generated evaluation artifacts, decode heatmap models when needed, and inspect representative validation predictions.

In [ ]:
from __future__ import annotations

import os
import sys
from pathlib import Path

os.environ.setdefault('TF_CPP_MIN_LOG_LEVEL', '2')

cwd = Path.cwd().resolve()
if (cwd / 'src').exists():
    project_root = cwd
elif (cwd.parent / 'src').exists():
    project_root = cwd.parent
else:
    raise RuntimeError('Run this notebook from the repository root or notebooks directory.')

os.chdir(project_root)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print(f'Project root: {project_root}')


In [ ]:
import json
from pathlib import Path

import keras
import matplotlib.pyplot as plt
import numpy as np

from src.data.freihand import FreiHand, SPLIT_SEED, SPLIT_VALIDATION_FRACTION
from src.evaluation.metrics import sample_mpke
from src.evaluation.overlays import GT_COLOR, PRED_COLOR, plot_keypoints


## Choose a run


In [ ]:
RUN_NAME = 'webcam-model'

checkpoint_path = project_root / 'models' / RUN_NAME / 'best.keras'
config_path = project_root / 'logs' / RUN_NAME / 'config.json'
history_path = project_root / 'logs' / RUN_NAME / 'history.json'
evaluation_path = project_root / 'artifacts' / RUN_NAME / 'evaluation.json'

missing = [path for path in (checkpoint_path, evaluation_path) if not path.exists()]
if missing:
    missing_text = chr(10).join(str(path) for path in missing)
    command = f'python -m src.evaluation.evaluate_run {RUN_NAME}'
    raise FileNotFoundError(f'Missing required files:{chr(10)}{missing_text}{chr(10)}Run first: {command}')

config = json.loads(config_path.read_text()) if config_path.exists() else {}
history = json.loads(history_path.read_text()) if history_path.exists() else {}
evaluation = json.loads(evaluation_path.read_text())
input_size = int(evaluation.get('input_size') or config.get('input_shape', [224])[0])

print(f'Run: {RUN_NAME}')
print(f'Model ID: {evaluation.get("model_id", config.get("model_id", config.get("model")))}')
print(f'Representation: {evaluation.get("representation", config.get("representation", ""))}')
print(f'Input size: {input_size}')
print(f'Validation MPKE: {evaluation["metrics"]["mpke_px"]:.2f} px')
if not history:
    print('Training history is not tracked; retrain locally to recreate training curves.')

## Load the checkpoint and prediction interface


In [ ]:
raw_model = keras.models.load_model(str(checkpoint_path))

if len(raw_model.output_shape) == 4:
    from src.models.heatmaps import wrap_with_keypoint_decoder

    eval_model = wrap_with_keypoint_decoder(raw_model, input_size=input_size)
    prediction_kind = 'heatmap decoded to keypoints'
else:
    eval_model = raw_model
    prediction_kind = 'direct coordinate regression'

rng = np.random.default_rng(0)
dummy_images = rng.random((2, input_size, input_size, 3), dtype=np.float32)
raw_output = raw_model.predict(dummy_images, verbose=0)
decoded_output = eval_model.predict(dummy_images, verbose=0)

def normalize_keypoints(array):
    array = np.asarray(array, dtype=np.float32)
    if array.ndim == 2:
        array = array.reshape(array.shape[0], 21, 2)
    return array

decoded_keypoints = normalize_keypoints(decoded_output)

print(f'Raw model output shape: {raw_output.shape}')
print(f'Prediction interface: {prediction_kind}')
print(f'Decoded keypoint shape: {decoded_keypoints.shape}')
print(f'Parameters: {raw_model.count_params():,}')

assert decoded_keypoints.shape == (2, 21, 2)


## Training curves


In [ ]:
if history:
    fig, axes = plt.subplots(1, 2, figsize=(9, 3.3))
    for ax, metric in zip(axes, ('loss', 'mae')):
        train = history.get(metric, [])
        val = history.get(f'val_{metric}', [])
        if train:
            ax.plot(range(1, len(train) + 1), train, label='train')
        if val:
            ax.plot(range(1, len(val) + 1), val, label='validation')
        ax.set_title(metric)
        ax.set_xlabel('epoch')
        ax.grid(axis='y', alpha=0.25)
        ax.legend(frameon=False)
    fig.suptitle(RUN_NAME)
    fig.tight_layout()
    plt.show()
else:
    print('No local training history found for this run.')

## Representative validation predictions


In [ ]:
dataset = FreiHand()
dataset.validate()
_, val_idx = dataset.train_validation_split(
    validation_fraction=SPLIT_VALIDATION_FRACTION,
    seed=SPLIT_SEED,
)

representative = evaluation['metrics'].get('representative_indices', {})
positions = [
    representative.get('best', 0),
    representative.get('median', len(val_idx) // 2),
    representative.get('worst', len(val_idx) - 1),
]
labels = ['best', 'median', 'worst']
sample_ids = val_idx[positions]

images, targets = dataset.load_batch(
    sample_ids,
    image_size=(input_size, input_size),
    normalize_images=True,
    flatten_keypoints=False,
)
predictions = normalize_keypoints(eval_model.predict(images, verbose=0))
errors = sample_mpke(predictions, targets)

fig, axes = plt.subplots(1, len(sample_ids), figsize=(3.2 * len(sample_ids), 3.3))
for ax, label, sample_id, image, target, prediction, error in zip(
    axes,
    labels,
    sample_ids,
    images,
    targets,
    predictions,
    errors,
):
    plot_keypoints(
        ax,
        image,
        target,
        prediction,
        gt_color=GT_COLOR,
        pred_color=PRED_COLOR,
        linewidth=1.1,
        marker_size=10,
    )
    ax.set_title(f'{label}: sample {int(sample_id)} ({error:.1f} px)')
axes[0].legend(loc='lower right', fontsize=8)
fig.tight_layout()
plt.show()
